In [1]:
from moabb.datasets import *
from moabb.paradigms import P300
import tensorly as tl

dataset = BNCI2014_008()
paradigm = P300(resample=32)
X, y, meta = paradigm.get_data(dataset, subjects=[1])
X = tl.tensor(X)
X.shape

(4200, 8, 32)

In [2]:
from bttda.hoda import BTTDA

bttda = BTTDA(
    ranks=[None]*8,
    hoda_params=dict(
        rank=None,
        shrinkage='lw',
        obj='tr',
        verbose=True,
        theta=0.2,
        toeplitz=(1,),       
    ),
    verbose=True,
    extra_train_info=True,

)
bttda.fit(X,y)

Fitting block 1/8...


Forward model :  22%|██▏       | 55/255 [00:00<00:00, 285.76it/s]


Fitting block 2/8...


Forward model :   5%|▍         | 12/255 [00:00<00:00, 296.16it/s]


Fitting block 3/8...


Forward model :   4%|▎         | 9/255 [00:00<00:00, 272.74it/s]


Fitting block 4/8...


Backward HODA model rank=(1, 4):  55%|█████▌    | 141/256 [00:24<00:20,  5.73it/s]

KeyboardInterrupt



In [ ]:
import pandas as pd

df = pd.DataFrame(bttda.train_info_)
df

In [ ]:
import plotly.express as px

if bttda.extra_train_info:
    fig = px.line(df, x='block', y='F_tr')
    fig.show()


In [ ]:
if bttda.extra_train_info:
    fig = px.line(df, x='block', y='F_rt')
    fig.show()


In [ ]:
if bttda.extra_train_info:
    fig = px.line(df, x='block', y='nmse')
    fig.show()


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from bttda.classification import SelectFCutoff

Xt = bttda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))

select = SelectFCutoff()
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'significant': select.scores_>1,
})
df

In [ ]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [ ]:
from sklearn.decomposition import PCA

x_viz = PCA(n_components=2, whiten=True).fit_transform(xts)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
  )

In [ ]:
from bttda.hoda import f_multiway

print(f"trace-ratio: {f_multiway(xts,y, method='tr')}")
print(f"ratio-trace: {f_multiway(xts,y, method='rt')}")